# 이화여대 Day0 vs Day12 | MW2 chromosome + pMW2 plasmid mapping

**Reference**: `NC_003923.1` (S. aureus MW2 chromosome, 2,820,462 bp) + `NC_005011.1` (pMW2 plasmid, 20,654 bp).  
**Inputs**: Day0.1, Day0.2, Day0.3, Day12.1, Day12.2, Day12.3 — PE150 FASTQ 12개.  
**Method**: BWA-MEM → samtools sort → GATK MarkDuplicates → haploid GATK HaplotypeCaller (GVCF) → six-sample joint genotyping → MW2 chrom/plasmid coverage → GenBank CDS-overlap annotation → Day0/Day12 candidate classification → optional assembly-to-pMW2 alignment.

**주의**: MW2 서열을 기준으로 확인한 변이는 기존 Day0.1 assembly의 VCF 좌표와 다릅니다. 직접 좌표 결합 금지. GATK haploid는 minor allele frequency 계산이 아니라 우세한 대립유전자 유전형 검출용입니다. 원본 FASTQ는 삭제하지 않습니다.

### 1. Google Drive 연결 및 데이터 확인

프로젝트의 원본 12개 FASTQ, 합쳐진 MW2 참조 FASTA, 공식 NCBI GenBank 주석을 확인합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib,json
root=Path('/content/drive/MyDrive/Day0_Day12_Variant_Calling')
samples=['Day0.1','Day0.2','Day0.3','Day12.1','Day12.2','Day12.3']
for s in samples:
    for mate in (1,2):
        f=root/'01.RawData'/s/f'{s}_L1_{mate}.fq.gz'
        assert f.is_file() and f.stat().st_size>10000, f'FASTQ missing/empty: {f}'
refdir=root/'outputs/mw2_reference'
fasta=refdir/'MW2_chromosome_plasmid.fasta'
assert fasta.is_file(),f'MW2 reference missing: {fasta}'
def sha256(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()
assert sha256(fasta)=='3a1ec8144b0c10cfd555afd5b3c3af0d15d30d521ee7efbfbb05e86bf87042eb','Reference hash mismatch'
for accession in ('NC_003923.1','NC_005011.1'):
    gb=refdir/f'{accession}.gb'
    assert gb.is_file(),f'GenBank annotation missing: {gb}'
print('INPUT CHECK PASS: 6 pairs (12 FASTQ)')
print('Reference SHA256 PASS:',sha256(fasta))
print('GATK MW2 results:',root/'outputs/mw2_gatk_v1')

### 2. 분석 프로그램 설치·버전 고정

GitHub Actions에서 합성 haploid VCF 테스트를 통과한 스크립트를 고정된 커밋에서 로드합니다. 원본 시료는 GitHub에 업로드하지 않습니다.

In [ ]:
import sys,subprocess,urllib.request
subprocess.run([sys.executable,'-m','pip','-q','install','pysam>=0.22,<1','biopython>=1.83,<2','pandas>=2,<3','matplotlib>=3,<4'],check=True)
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','minimap2'],check=True)
base='https://raw.githubusercontent.com/kimtk94/Codex/268c21e81c936a620dc1511c1db947c09eb43865/research/ewha_day0_day12'
for name in ('gatk_haploid_pipeline.py','mw2_mapping_pipeline.py'):
    url=base+'/'+name
    dst='/content/'+name
    urllib.request.urlretrieve(url,dst)
    subprocess.run([sys.executable,'-m','py_compile',dst],check=True)
print('MW2 pipeline scripts loaded and compiled')
subprocess.run([sys.executable,'/content/mw2_mapping_pipeline.py','--help'],check=True)

### 3. Day0.1 파일럿

1개 샘플의 정렬·중복표시·GVCF·염색체/플라스미드 커버리지·assembly-pMW2 비교를 먼저 수행합니다. 파일럿 중 오류가 나면 전체 샘플로 넘어가지 않습니다.

In [ ]:
subprocess.run([sys.executable,'/content/mw2_mapping_pipeline.py','--pilot'],check=True)
pilot=root/'outputs/mw2_gatk_v1/coverage/Day0.1.json'
print(json.dumps(json.loads(pilot.read_text()),ensure_ascii=False,indent=2))

### 4. 전체 6개 샘플 MW2 mapping + joint GATK variant calling

파일럿에서 검증된 GVCF와 BAM은 재사용하고, 나머지 5개 샘플을 분석합니다. 결과는 Google Drive `outputs/mw2_gatk_v1/`에 저장됩니다. 계속 실행 여부는 사용자가 이 셀을 실행하여 결정합니다.

In [ ]:
subprocess.run([sys.executable,'/content/mw2_mapping_pipeline.py'],check=True)
print('Finished. Results:',root/'outputs/mw2_gatk_v1')

### 5. 통계·후보 변이 결과

- `coverage/*.json`: 6개 샘플별 chromosome/pMW2 1×/10×/30× breadth, 평균/중앙 depth
- `bams/*.bam` + `.bai`: MW2 매핑·IGV용 중복표시 BAM
- `samples/*/*.g.vcf.gz`: 샘플별 GVCF
- `cohort/joint.normalized.vcf.gz`: 공동 호출·정규화 VCF
- `cohort/mw2_variant_candidates.csv`: 6샘플 genotype/depth/gene overlap 및 Day12 후보 분류
- `assembly_plasmid/*.pMW2.paf`: 조립서열과 pMW2의 유사성 screening

변이 후보는 **Day0의 호출 가능 여부, Day12 반복 재현성, 별도 pMW2 coverage**를 함께 사용하여 분류합니다. 내성 표현형이나 신규 플라스미드 획득은 이 분석만으로 확정할 수 없습니다.

In [ ]:
import pandas as pd,json,matplotlib.pyplot as plt
from IPython.display import display
out=root/'outputs/mw2_gatk_v1'
summary=json.loads((out/'cohort/mw2_summary.json').read_text())
print('Status:',summary['status'],'Reference:',summary['reference_contigs'])
display(pd.Series(summary['candidate_counts'],name='allele_count').to_frame())
rows=[]
for sample in samples:
    report=json.loads((out/'coverage'/f'{sample}.json').read_text())['coverage']
    for acc in ('NC_003923.1','NC_005011.1'):
        x=report[acc]
        rows.append({'sample':sample,'reference':acc,'breadth_10x':x['breadth_10x'],
                     'mean_depth':x['mean_depth'],'median_depth':x['median_depth']})
cov=pd.DataFrame(rows)
display(cov)
df=pd.read_csv(out/'cohort/mw2_variant_candidates.csv')
display(df[df['classification']=='DAY12_REPLICATED_CANDIDATE'].head(100))
print('Full variant table:',out/'cohort/mw2_variant_candidates.csv')

### 검증 해석

- `NC_003923.1`: 세균 염색체 상 공통 및 Day12 특이적 SNP/Indel을 MW2 좌표로 표기
- `NC_005011.1`: pMW2 플라스미드 read mapping breadth/depth, assembly alignment, 관련 CDS annotation 확인
- 기존 `Day0.1` assembly VCF와 직접 SNP 위치 병합 불가. MW2 좌표 변환 또는 독립 교차매핑 후 비교 필요
- Low-AF 후보는 기존 LoFreq v9.1과 좌표계가 다르므로 MW2 reference로 LoFreq 재호출이 필요
- GATK 공동변이 PASS는 샘플의 내성 변화나 획득 시점을 의미하지 않음.